In [8]:
import yaml, os

with open('/content/PlantDoc-2/data.yaml') as f:
    cfg = yaml.safe_load(f)

# No valid split exists — use test as val
cfg['train'] = '/content/PlantDoc-2/train/images'
cfg['val']   = '/content/PlantDoc-2/test/images'
cfg['test']  = '/content/PlantDoc-2/test/images'

with open('/content/PlantDoc-2/data.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

# Confirm
with open('/content/PlantDoc-2/data.yaml') as f:
    print(f.read())

print("train exists:", os.path.exists('/content/PlantDoc-2/train/images'))
print("test exists :", os.path.exists('/content/PlantDoc-2/test/images'))

names:
- Apple Scab Leaf
- Apple leaf
- Apple rust leaf
- Bell_pepper leaf spot
- Bell_pepper leaf
- Blueberry leaf
- Cherry leaf
- Corn Gray leaf spot
- Corn leaf blight
- Corn rust leaf
- Peach leaf
- Potato leaf early blight
- Potato leaf late blight
- Potato leaf
- Raspberry leaf
- Soyabean leaf
- Soybean leaf
- Squash Powdery mildew leaf
- Strawberry leaf
- Tomato Early blight leaf
- Tomato Septoria leaf spot
- Tomato leaf bacterial spot
- Tomato leaf late blight
- Tomato leaf mosaic virus
- Tomato leaf yellow virus
- Tomato leaf
- Tomato mold leaf
- Tomato two spotted spider mites leaf
- grape leaf black rot
- grape leaf
nc: 30
roboflow:
  license: CC BY 4.0
  project: plantdoc
  url: https://universe.roboflow.com/joseph-nelson/plantdoc/dataset/2
  version: 2
  workspace: joseph-nelson
test: /content/PlantDoc-2/test/images
train: /content/PlantDoc-2/train/images
val: /content/PlantDoc-2/test/images

train exists: True
test exists : True


In [9]:


"""
============================================================
 YOLO PLANT DISEASE DETECTOR  —  IMPROVED TRAINING SCRIPT
============================================================
Fixes & improvements over original:
  • YOLOv8s (small) instead of nano — better accuracy,
    still runs on Colab T4 GPU
  • Proper hyperparameter config passed explicitly
  • Mosaic, mixup, copy-paste augmentation enabled
  • Cosine LR decay instead of default step decay
  • Early stopping with patience=15
  • Confidence threshold raised to 0.20 at inference
    (original 0.15 caused noisy crops into CLIP)
  • Full evaluation: mAP@50, mAP@50-95, per-class AP
  • Confusion matrix + PR curve saved automatically
  • Crash-safe: checkpoint resume handled cleanly
  • All 30 class names verified against data.yaml
  • NMS IoU threshold tuned (0.45 instead of default 0.7)
    to reduce duplicate detections on overlapping lesions
============================================================
Run order in Colab:
  Step 1 → Train from scratch (or resume if crashed)
  Step 2 → Evaluate best.pt on test set
  Step 3 → Export + verify class names match inference pipeline
============================================================
"""

# ── 0. Install ────────────────────────────────────────────
!pip install ultralytics roboflow -q

import os, random, shutil
from pathlib import Path

import torch
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
from PIL import Image
from ultralytics import YOLO

# ── 1. CONFIG  (edit these) ───────────────────────────────
SAVE_DIR      = "/content/drive/MyDrive/PlantDoc_Project_2026"
PROJECT_DIR   = f"{SAVE_DIR}/YOLO_Training"
DATA_YAML     = "/content/PlantDoc-2/data.yaml"   # from Roboflow download
RESUME_CKPT   = None   # set to path like f"{PROJECT_DIR}/train/weights/last.pt"
                        # if a previous run crashed

MODEL_SIZE    = "yolov8s.pt"   # s=small (better than n=nano, still fast)
EPOCHS        = 100            # with early stopping patience=15
IMG_SIZE      = 640
BATCH_SIZE    = 16             # safe for Colab T4; increase to 32 on A100
WORKERS       = 2
CONF_THRESH   = 0.20           # inference confidence (raised from 0.15)
IOU_THRESH    = 0.45           # NMS IoU (lower = fewer duplicate boxes)
SEED          = 42

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f"{SAVE_DIR}/eval_results", exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
print(f"Model  : {MODEL_SIZE}")

# ── 2. DOWNLOAD DATASET (if not already done) ─────────────
def download_dataset():
    """Download PlantDoc v2 from Roboflow in YOLOv8 format."""
    from roboflow import Roboflow
    rf      = Roboflow(api_key="hRZ31vcG96RlYJoYR2EK")
    project = rf.workspace("joseph-nelson").project("plantdoc")
    version = project.version(2)
    dataset = version.download("yolov8")
    print(f"Dataset location: {dataset.location}")
    return dataset.location

if not os.path.exists("/content/PlantDoc-2"):
    print("Downloading PlantDoc dataset...")
    dataset_loc = download_dataset()
else:
    dataset_loc = "/content/PlantDoc-2"
    print(f"Dataset already exists at {dataset_loc}")

# Verify data.yaml exists
assert os.path.exists(DATA_YAML), f"data.yaml not found at {DATA_YAML}"
print(f"data.yaml found at {DATA_YAML}")

# ── 3. INSPECT + VALIDATE DATA.YAML ──────────────────────
import yaml

with open(DATA_YAML) as f:
    data_cfg = yaml.safe_load(f)

print("\n=== data.yaml contents ===")
print(f"  train : {data_cfg.get('train')}")
print(f"  val   : {data_cfg.get('val')}")
print(f"  test  : {data_cfg.get('test', 'not specified')}")
print(f"  nc    : {data_cfg.get('nc')} classes")
print(f"  names :")
for i, name in enumerate(data_cfg.get("names", [])):
    print(f"    {i:>2}: {name}")

# Expected 30 classes — warn if mismatch
EXPECTED_CLASSES = [
    "Apple Scab Leaf", "Apple leaf", "Apple rust leaf",
    "Bell_pepper leaf spot", "Bell_pepper leaf", "Blueberry leaf",
    "Cherry leaf", "Corn Gray leaf spot", "Corn leaf blight",
    "Corn rust leaf", "Peach leaf", "Potato leaf early blight",
    "Potato leaf late blight", "Potato leaf", "Raspberry leaf",
    "Soyabean leaf", "Soybean leaf", "Squash Powdery mildew leaf",
    "Strawberry leaf", "Tomato Early blight leaf",
    "Tomato Septoria leaf spot", "Tomato leaf bacterial spot",
    "Tomato leaf late blight", "Tomato leaf mosaic virus",
    "Tomato leaf yellow virus", "Tomato leaf", "Tomato mold leaf",
    "Tomato two spotted spider mites leaf", "grape leaf black rot",
    "grape leaf",
]

yaml_names = data_cfg.get("names", [])
missing_in_yaml = [c for c in EXPECTED_CLASSES if c not in yaml_names]
extra_in_yaml   = [c for c in yaml_names if c not in EXPECTED_CLASSES]
if missing_in_yaml:
    print(f"\n  [WARN] Classes in expected list but NOT in yaml: {missing_in_yaml}")
if extra_in_yaml:
    print(f"  [WARN] Classes in yaml but NOT in expected list : {extra_in_yaml}")
if not missing_in_yaml and not extra_in_yaml:
    print("\n  All class names match expected list.")

# ── 4. CLASS DISTRIBUTION ANALYSIS ───────────────────────
def count_labels(images_dir, labels_dir, class_names):
    """Count per-class instances in a split."""
    counts = {i: 0 for i in range(len(class_names))}
    label_files = list(Path(labels_dir).glob("*.txt"))
    for lf in label_files:
        with open(lf) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    cls_id = int(parts[0])
                    counts[cls_id] = counts.get(cls_id, 0) + 1
    return counts

train_labels_dir = str(Path(DATA_YAML).parent / "train" / "labels")
test_labels_dir  = str(Path(DATA_YAML).parent / "test"  / "labels")

if os.path.exists(train_labels_dir):
    train_counts = count_labels(
        str(Path(DATA_YAML).parent / "train" / "images"),
        train_labels_dir, yaml_names
    )
    print("\n=== Train class distribution ===")
    total_instances = sum(train_counts.values())
    for i, name in enumerate(yaml_names):
        cnt  = train_counts.get(i, 0)
        bar  = "█" * int(30 * cnt / max(train_counts.values(), default=1))
        print(f"  {i:>2} {name:<45} {cnt:>4}  {bar}")
    print(f"\n  Total instances: {total_instances}")
    print(f"  Total images   : {len(list(Path(train_labels_dir).glob('*.txt')))}")

# ── 5. CUSTOM HYPERPARAMETERS ─────────────────────────────
# These override Ultralytics defaults for plant disease detection
HYPS = {
    # Learning rate
    "lr0":        0.01,    # initial LR
    "lrf":        0.01,    # final LR = lr0 * lrf (cosine decay)
    "momentum":   0.937,
    "weight_decay": 5e-4,

    # Warmup
    "warmup_epochs":   3.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr":  0.1,

    # Loss weights
    "box":  7.5,
    "cls":  0.5,
    "dfl":  1.5,

    # Augmentation — aggressive for plant disease variability
    "hsv_h":      0.015,   # hue jitter
    "hsv_s":      0.7,     # saturation jitter
    "hsv_v":      0.4,     # value jitter
    "degrees":    10.0,    # rotation
    "translate":  0.1,
    "scale":      0.5,
    "shear":      2.0,
    "perspective": 0.0,
    "flipud":     0.3,     # vertical flip (useful for leaves)
    "fliplr":     0.5,
    "mosaic":     1.0,     # mosaic augmentation ON
    "mixup":      0.1,     # mild mixup
    "copy_paste": 0.1,     # copy-paste augmentation
}

# ── 6. TRAIN ──────────────────────────────────────────────
def train_yolo():
    if RESUME_CKPT and os.path.exists(RESUME_CKPT):
        # Resume from crash checkpoint
        print(f"\nResuming from {RESUME_CKPT}")
        model   = YOLO(RESUME_CKPT)
        results = model.train(resume=True)
    else:
        print(f"\nStarting fresh training with {MODEL_SIZE}")
        model   = YOLO(MODEL_SIZE)
        results = model.train(
            data      = DATA_YAML,
            epochs    = EPOCHS,
            imgsz     = IMG_SIZE,
            batch     = BATCH_SIZE,
            workers   = WORKERS,
            device    = device,
            project   = PROJECT_DIR,
            name      = "train",
            exist_ok  = True,
            # Optimization
            optimizer = "AdamW",
            cos_lr    = True,       # cosine LR decay
            # Regularization
            dropout   = 0.0,
            label_smoothing = 0.05, # mild label smoothing
            # Stopping
            patience  = 15,         # early stopping
            # NMS
            iou       = IOU_THRESH,
            conf      = CONF_THRESH,
            # Augmentation hyperparameters
            **HYPS,
            # Logging
            plots     = True,
            save      = True,
            save_period = 10,       # save checkpoint every 10 epochs
            verbose   = True,
            seed      = SEED,
        )
    return model, results

model, train_results = train_yolo()

best_weights = f"{PROJECT_DIR}/train/weights/best.pt"
print(f"\nBest weights: {best_weights}")

# Copy best weights to SAVE_DIR for use in inference pipeline
dest = f"{SAVE_DIR}/Plantdoc_Yolo_v1_best.pt"
shutil.copy2(best_weights, dest)
print(f"Copied best.pt → {dest}")

# ── 7. EVALUATE ON TEST SET ───────────────────────────────
print("\n===== EVALUATING ON TEST SET =====")
best_model = YOLO(best_weights)

# val() on the test split
test_results = best_model.val(
    data    = DATA_YAML,
    split   = "test",           # use test split
    imgsz   = IMG_SIZE,
    batch   = BATCH_SIZE,
    conf    = CONF_THRESH,
    iou     = IOU_THRESH,
    device  = device,
    plots   = True,
    save_json = True,
    project = f"{SAVE_DIR}/eval_results",
    name    = "yolo_test_eval",
)

# Print key metrics
print("\n=== Test Set Metrics ===")
metrics = test_results.results_dict
print(f"  mAP@50      : {metrics.get('metrics/mAP50(B)',    0):.4f}")
print(f"  mAP@50-95   : {metrics.get('metrics/mAP50-95(B)', 0):.4f}")
print(f"  Precision   : {metrics.get('metrics/precision(B)', 0):.4f}")
print(f"  Recall      : {metrics.get('metrics/recall(B)',    0):.4f}")

# Per-class AP
print("\n=== Per-class AP@50 ===")
if hasattr(test_results, 'ap_class_index') and test_results.ap_class_index is not None:
    ap50_per_class = test_results.box.ap50   # array of AP@50 per class
    class_indices  = test_results.ap_class_index
    for idx, ap in zip(class_indices, ap50_per_class):
        cls_name = yaml_names[idx] if idx < len(yaml_names) else f"class_{idx}"
        bar = "█" * int(20 * ap)
        print(f"  {cls_name:<45} AP@50={ap:.3f}  {bar}")
else:
    print("  (Per-class AP not available from this Ultralytics version)")

# ── 8. VISUAL SPOT-CHECK ──────────────────────────────────
def spot_check(n_images=6, split="test"):
    """Show n random predictions with bounding boxes."""
    images_dir = str(Path(DATA_YAML).parent / split / "images")
    all_imgs   = list(Path(images_dir).glob("*.jpg")) + \
                 list(Path(images_dir).glob("*.png"))

    if not all_imgs:
        print("No images found for spot check.")
        return

    sample = random.sample(all_imgs, min(n_images, len(all_imgs)))
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()

    for ax, img_path in zip(axes, sample):
        results  = best_model.predict(
            str(img_path), conf=CONF_THRESH, iou=IOU_THRESH, verbose=False)[0]
        img_rgb  = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb)

        for box in results.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            cls_id   = int(box.cls[0])
            conf     = float(box.conf[0])
            cls_name = yaml_names[cls_id] if cls_id < len(yaml_names) else str(cls_id)
            # Short label for readability
            short    = cls_name.replace(" leaf", "").replace("_", " ")[:25]
            rect = patches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=2, edgecolor="lime", facecolor="none"
            )
            ax.add_patch(rect)
            ax.text(x1, max(y1-6, 0), f"{short} {conf:.2f}",
                    color="white", fontsize=7,
                    bbox=dict(boxstyle="round,pad=0.2", facecolor="green",
                              alpha=0.8, edgecolor="none"))

        ax.set_title(Path(img_path).name[:30], fontsize=8)
        ax.axis("off")

    plt.suptitle(f"YOLO Spot-Check ({split} split, conf≥{CONF_THRESH})",
                 fontsize=12, y=1.01)
    plt.tight_layout()
    out_path = f"{SAVE_DIR}/eval_results/spot_check_{split}.png"
    fig.savefig(out_path, dpi=120, bbox_inches="tight")
    print(f"\nSpot-check saved → {out_path}")

spot_check(n_images=6, split="test")

# ── 9. DETECTION MISS ANALYSIS ────────────────────────────
def analyze_misses(split="test", max_show=20):
    """Find images where YOLO detects nothing at all."""
    images_dir = str(Path(DATA_YAML).parent / split / "images")
    labels_dir = str(Path(DATA_YAML).parent / split / "labels")
    all_imgs   = list(Path(images_dir).glob("*.jpg")) + \
                 list(Path(images_dir).glob("*.png"))

    missed       = []
    total        = 0
    has_gt_count = 0

    for img_path in all_imgs:
        label_file = Path(labels_dir) / (img_path.stem + ".txt")
        if not label_file.exists():
            continue
        with open(label_file) as f:
            lines = [l.strip() for l in f if l.strip()]
        if not lines:
            continue
        has_gt_count += 1
        total        += 1

        results = best_model.predict(
            str(img_path), conf=CONF_THRESH, verbose=False)[0]
        if len(results.boxes) == 0:
            gt_cls = int(lines[0].split()[0])
            missed.append((str(img_path), gt_cls))

    miss_rate = len(missed) / max(total, 1)
    print(f"\n=== Miss Analysis ({split}) ===")
    print(f"  Images with ground-truth : {has_gt_count}")
    print(f"  YOLO detected nothing    : {len(missed)}")
    print(f"  Miss rate                : {100*miss_rate:.1f}%")

    if missed:
        miss_by_class = {}
        for _, cls_id in missed:
            name = yaml_names[cls_id] if cls_id < len(yaml_names) else str(cls_id)
            miss_by_class[name] = miss_by_class.get(name, 0) + 1
        print("  Missed instances by class:")
        for name, cnt in sorted(miss_by_class.items(), key=lambda x: -x[1])[:max_show]:
            print(f"    {name:<45} {cnt}")

    return missed

missed_images = analyze_misses(split="test")

# ── 10. EXPORT MODEL INFO ─────────────────────────────────
print("\n=== Model Summary ===")
best_model.info(verbose=True)

print(f"\nAll done!")
print(f"  Best weights  : {dest}")
print(f"  Eval results  : {SAVE_DIR}/eval_results/")
print(f"\nNext steps:")
print(f"  1. Run clip_dcon_trainer.py  to train the adapter + build feature bank")
print(f"  2. Run dino_species_trainer.py  to train the species classifier")
print(f"  3. Run inference_pipeline.py  for end-to-end evaluation")


Device : cuda
Model  : yolov8s.pt
Dataset already exists at /content/PlantDoc-2
data.yaml found at /content/PlantDoc-2/data.yaml

=== data.yaml contents ===
  train : /content/PlantDoc-2/train/images
  val   : /content/PlantDoc-2/test/images
  test  : /content/PlantDoc-2/test/images
  nc    : 30 classes
  names :
     0: Apple Scab Leaf
     1: Apple leaf
     2: Apple rust leaf
     3: Bell_pepper leaf spot
     4: Bell_pepper leaf
     5: Blueberry leaf
     6: Cherry leaf
     7: Corn Gray leaf spot
     8: Corn leaf blight
     9: Corn rust leaf
    10: Peach leaf
    11: Potato leaf early blight
    12: Potato leaf late blight
    13: Potato leaf
    14: Raspberry leaf
    15: Soyabean leaf
    16: Soybean leaf
    17: Squash Powdery mildew leaf
    18: Strawberry leaf
    19: Tomato Early blight leaf
    20: Tomato Septoria leaf spot
    21: Tomato leaf bacterial spot
    22: Tomato leaf late blight
    23: Tomato leaf mosaic virus
    24: Tomato leaf yellow virus
    25: Tomato 

In [10]:
from google.colab import files

# Download the model weights
files.download('/content/drive/MyDrive/PlantDoc_Project_2026/Plantdoc_Yolo_v1_best.pt')

# Download eval results
files.download('/content/drive/MyDrive/PlantDoc_Project_2026/eval_results/spot_check_test.png')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>